## EE8223 Deep Learning Project

## Extraction of Wav2Vec2-Based Audio Embeddings from IEMOCAP Dataset

#Student: Jason Yip

### Overview
This script extracts frame-level audio embeddings from the IEMOCAP dataset using a fine-tuned Wav2Vec2 model. The embeddings are pooled to generate a fixed-length vector for each audio file, which is then saved alongside metadata for downstream tasks like emotion recognition.

### Detailed Steps

1. **Extract Dataset**:  
   - Unzips the IEMOCAP dataset and prepares the directory structure for processing.

2. **Load Metadata**:  
   - Reads the CSV file containing metadata (e.g., file paths and emotion labels).  
   - Filters rows to include only audio files available in the specified "train" folder.

3. **Load Pretrained Model and Processor**:  
   - Loads a fine-tuned Wav2Vec2 model and its processor for feature extraction.  
   - Ensures the model is set to evaluation mode and uses GPU if available.

4. **Audio Processing and Embedding Extraction**:  
   - Loads each WAV file with `librosa` at a 16 kHz sample rate.  
   - Processes the audio input using the Wav2Vec2 processor to prepare it for the model.  
   - Extracts frame-level embeddings from the Wav2Vec2 model and performs mean pooling to obtain a single vector representation for each audio file.

5. **Store Embeddings and Metadata**:  
   - Collects embeddings, file paths, labels (emotion), and metadata into a list.  
   - Converts the list to a DataFrame for structured storage.

6. **Save Results**:  
   - Saves the embeddings and metadata as a CSV file to Google Drive, making it accessible for further analysis or model training.

### Key Features

- **End-to-End Embedding Extraction**: Automates the process of extracting meaningful audio embeddings from raw WAV files.  
- **Emotion Label Association**: Preserves metadata such as emotion labels for use in emotion recognition tasks.  
- **Seamless Integration with Fine-Tuned Models**: Leverages a fine-tuned Wav2Vec2 model to generate task-specific embeddings.  
- **Scalable and Efficient**: Processes large audio datasets with GPU acceleration and batch-wise operations.  
- **Reproducible Output**: Saves embeddings and metadata in a CSV format for consistency and reuse in downstream tasks.

This script is ideal for researchers and practitioners working on tasks such as speech emotion recognition or other audio-based machine learning applications.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Training Set

This script extracts audio embeddings from the training set of the IEMOCAP dataset using a fine-tuned Wav2Vec2 model. It processes the audio files, extracts meaningful embeddings through mean pooling, and saves the results along with metadata for downstream analysis, such as emotion recognition.

In [ ]:
import os
import zipfile
import librosa
import torch
import pandas as pd
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification  # Use the fine-tuned model class

# Define path for the new zip file and extraction
zip_path = '/content/drive/MyDrive/Copy of IEMOCAP_new.zip'
extract_path = '/content/iemocap_data'

# Extract Copy of IEMOCAP_new.zip if not already extracted
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# Define paths to the CSV and the "train" folder inside the extracted "IEMOCAP_Copied" folder
csv_path = os.path.join(extract_path, 'Copy of IEMOCAP', 'IEMOCAP_Copied', 'iemocap_full_dataset.csv')
train_folder = os.path.join(extract_path, 'Copy of IEMOCAP', 'IEMOCAP_Copied', 'train')

# Ensure the directory exists
os.makedirs('/content/drive/MyDrive/IEMOCAP_embeddings_NEW_bce', exist_ok=True)

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the CSV file
data = pd.read_csv(csv_path)

# List all files in the train folder and get their base filenames
train_files = {filename.replace('.wav', '') for filename in os.listdir(train_folder) if filename.endswith('.wav')}

# Filter the CSV to include only rows corresponding to files in the train folder
train_data = data[data['path'].apply(lambda x: x.replace('/', '_').replace('.wav', '') in train_files)].reset_index(drop=True)

# Load the fine-tuned wav2vec2 model and processor from a specific checkpoint or final model
checkpoints_path = "/content/drive/MyDrive/Wav2Vec2_Checkpoints_bce/checkpoint-5630"
processor_save_path = "/content/drive/MyDrive/Wav2Vec2_Processor_ASVspoof_bce"
processor = Wav2Vec2Processor.from_pretrained(processor_save_path)
model = Wav2Vec2ForSequenceClassification.from_pretrained(checkpoints_path)
model = model.to(device)  # Ensure model is on GPU if available
model.eval()  # Set to evaluation mode

# Function to load audio, process it, and extract embeddings
def extract_embeddings(file_path):
    try:
        # Load the audio file with librosa at a sample rate of 16 kHz
        audio_input, _ = librosa.load(file_path, sr=16000)
    except Exception as e:
        print(f"Warning: Could not load {file_path}. Skipping. Error: {e}")
        return None, None

    # Prepare the input values for the fine-tuned wav2vec2 model
    input_values = processor(audio_input, sampling_rate=16000, return_tensors="pt", padding=True).input_values.to(device)

    # Extract embeddings using the model
    with torch.no_grad():
        frame_embeddings = model.wav2vec2(input_values).last_hidden_state  # Access wav2vec2 model part directly

    # Mean pooling across frames to get a single embedding vector per audio file
    pooled_embedding = frame_embeddings.mean(dim=1).squeeze()

    # Return the embedding and its shape
    return pooled_embedding.cpu().numpy(), pooled_embedding.shape

# Initialize list to store embeddings
embeddings_list = []

# Process files in the filtered train data order
for _, row in tqdm(train_data.iterrows(), total=len(train_data)):
    base_filename = row['path'].replace('/', '_').replace('.wav', '')
    file_path = os.path.join(train_folder, base_filename + ".wav")

    if not os.path.exists(file_path):
        continue

    # Extract the emotion label
    emotion = row['emotion']

    # Extract embeddings from the WAV file
    embeddings, embedding_shape = extract_embeddings(file_path)

    if embeddings is None:
        continue

    # Append embeddings, file path, label, and shape info to list
    embeddings_list.append({
        "file_path": base_filename,
        "folder": "train",
        "emotion_label": emotion,
        "embedding_shape": embedding_shape,
        "embedding": embeddings
    })

# Convert list to DataFrame
embeddings_df = pd.DataFrame(embeddings_list)

# Convert embeddings to lists for saving in a CSV
embeddings_df['embedding'] = embeddings_df['embedding'].apply(lambda x: x.tolist())

# Save the DataFrame to Google Drive
embeddings_csv_path = "/content/drive/MyDrive/IEMOCAP_embeddings_NEW_bce/iemocap_embeddings_train_NEW_bce.csv"
embeddings_df.to_csv(embeddings_csv_path, index=False)

# Display the DataFrame in Colab
print(embeddings_df)
print(f"Embeddings and metadata saved to Google Drive at {embeddings_csv_path}")


Using device: cuda


100%|██████████| 7027/7027 [09:11<00:00, 12.75it/s]


                                              file_path folder emotion_label  \
0     Session1_sentences_wav_Ses01F_script02_1_Ses01...  train           neu   
1     Session1_sentences_wav_Ses01F_script02_1_Ses01...  train           xxx   
2     Session1_sentences_wav_Ses01F_script02_1_Ses01...  train           neu   
3     Session1_sentences_wav_Ses01F_script02_1_Ses01...  train           xxx   
4     Session1_sentences_wav_Ses01F_script02_1_Ses01...  train           ang   
...                                                 ...    ...           ...   
7022  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  train           sad   
7023  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  train           neu   
7024  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  train           neu   
7025  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  train           neu   
7026  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  train           neu   

     embedding_shape                   

## Validation Set

This script extracts audio embeddings from the validation set of the IEMOCAP dataset using a fine-tuned Wav2Vec2 model. It processes the audio files, extracts meaningful embeddings through mean pooling, and saves the results along with metadata for downstream analysis, such as emotion recognition.

In [ ]:
import os
import zipfile
import librosa
import torch
import pandas as pd
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification  # Use the fine-tuned model class

# Define path for the new zip file and extraction
zip_path = '/content/drive/MyDrive/Copy of IEMOCAP_new.zip'
extract_path = '/content/iemocap_data'

# Extract Copy of IEMOCAP_new.zip if not already extracted
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# Define paths to the CSV and the "validate" folder inside the extracted "IEMOCAP_Copied" folder
csv_path = os.path.join(extract_path, 'Copy of IEMOCAP', 'IEMOCAP_Copied', 'iemocap_full_dataset.csv')
validate_folder = os.path.join(extract_path, 'Copy of IEMOCAP', 'IEMOCAP_Copied', 'validate')

# Ensure the directory exists
os.makedirs('/content/drive/MyDrive/IEMOCAP_embeddings_NEW_bce', exist_ok=True)

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the CSV file
data = pd.read_csv(csv_path)

# List all files in the validate folder and get their base filenames
validate_files = {filename.replace('.wav', '') for filename in os.listdir(validate_folder) if filename.endswith('.wav')}

# Filter the CSV to include only rows corresponding to files in the validate folder
validate_data = data[data['path'].apply(lambda x: x.replace('/', '_').replace('.wav', '') in validate_files)].reset_index(drop=True)

# Load the fine-tuned wav2vec2 model and processor from a specific checkpoint or final model
checkpoints_path = "/content/drive/MyDrive/Wav2Vec2_Checkpoints_bce/checkpoint-5630"
processor_save_path = "/content/drive/MyDrive/Wav2Vec2_Processor_ASVspoof_bce"
processor = Wav2Vec2Processor.from_pretrained(processor_save_path)
model = Wav2Vec2ForSequenceClassification.from_pretrained(checkpoints_path)
model = model.to(device)  # Ensure model is on GPU if available
model.eval()  # Set to evaluation mode

# Function to load audio, process it, and extract embeddings
def extract_embeddings(file_path):
    try:
        # Load the audio file with librosa at a sample rate of 16 kHz
        audio_input, _ = librosa.load(file_path, sr=16000)
    except Exception as e:
        print(f"Warning: Could not load {file_path}. Skipping. Error: {e}")
        return None, None

    # Prepare the input values for the fine-tuned wav2vec2 model
    input_values = processor(audio_input, sampling_rate=16000, return_tensors="pt", padding=True).input_values.to(device)

    # Extract embeddings using the model
    with torch.no_grad():
        frame_embeddings = model.wav2vec2(input_values).last_hidden_state  # Access wav2vec2 model part directly

    # Mean pooling across frames to get a single embedding vector per audio file
    pooled_embedding = frame_embeddings.mean(dim=1).squeeze()

    # Return the embedding and its shape
    return pooled_embedding.cpu().numpy(), pooled_embedding.shape

# Initialize list to store embeddings
embeddings_list = []

# Process files in the filtered validate data order
for _, row in tqdm(validate_data.iterrows(), total=len(validate_data)):
    base_filename = row['path'].replace('/', '_').replace('.wav', '')
    file_path = os.path.join(validate_folder, base_filename + ".wav")

    if not os.path.exists(file_path):
        continue

    # Extract the emotion label
    emotion = row['emotion']

    # Extract embeddings from the WAV file
    embeddings, embedding_shape = extract_embeddings(file_path)

    if embeddings is None:
        continue

    # Append embeddings, file path, label, and shape info to list
    embeddings_list.append({
        "file_path": base_filename,
        "folder": "validate",
        "emotion_label": emotion,
        "embedding_shape": embedding_shape,
        "embedding": embeddings
    })

# Convert list to DataFrame
embeddings_df = pd.DataFrame(embeddings_list)

# Convert embeddings to lists for saving in a CSV
embeddings_df['embedding'] = embeddings_df['embedding'].apply(lambda x: x.tolist())

# Save the DataFrame to Google Drive
embeddings_csv_path = "/content/drive/MyDrive/IEMOCAP_embeddings_NEW_bce/iemocap_embeddings_validate_NEW_bce.csv"
embeddings_df.to_csv(embeddings_csv_path, index=False)

# Display the DataFrame in Colab
print(embeddings_df)
print(f"Embeddings and metadata saved to Google Drive at {embeddings_csv_path}")


Using device: cuda


100%|██████████| 1505/1505 [01:58<00:00, 12.75it/s]


                                              file_path    folder  \
0     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
1     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
2     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
3     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
4     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
...                                                 ...       ...   
1500  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   
1501  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   
1502  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   
1503  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   
1504  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   

     emotion_label embedding_shape  \
0              fru         (1024,)   
1              neu         (1024,)   
2              neu         (1024,)   
3              neu 

## Test Set
This script extracts audio embeddings from the test set of the IEMOCAP dataset using a fine-tuned Wav2Vec2 model. It processes the audio files, extracts meaningful embeddings through mean pooling, and saves the results along with metadata for downstream analysis, such as emotion recognition.

In [ ]:
import os
import zipfile
import librosa
import torch
import pandas as pd
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification  # Use the fine-tuned model class

# Define path for the new zip file and extraction
zip_path = '/content/drive/MyDrive/Copy of IEMOCAP_new.zip'
extract_path = '/content/iemocap_data'

# Extract Copy of IEMOCAP_new.zip if not already extracted
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# Define paths to the CSV and the "test" folder inside the extracted "IEMOCAP_Copied" folder
csv_path = os.path.join(extract_path, 'Copy of IEMOCAP', 'IEMOCAP_Copied', 'iemocap_full_dataset.csv')
test_folder = os.path.join(extract_path, 'Copy of IEMOCAP', 'IEMOCAP_Copied', 'test')

# Ensure the directory exists
os.makedirs('/content/drive/MyDrive/IEMOCAP_embeddings_NEW_bce', exist_ok=True)

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the CSV file
data = pd.read_csv(csv_path)

# List all files in the test folder and get their base filenames
test_files = {filename.replace('.wav', '') for filename in os.listdir(test_folder) if filename.endswith('.wav')}

# Filter the CSV to include only rows corresponding to files in the test folder
test_data = data[data['path'].apply(lambda x: x.replace('/', '_').replace('.wav', '') in test_files)].reset_index(drop=True)

# Load the fine-tuned wav2vec2 model and processor from a specific checkpoint or final model
checkpoints_path = "/content/drive/MyDrive/Wav2Vec2_Checkpoints_bce/checkpoint-5630"
processor_save_path = "/content/drive/MyDrive/Wav2Vec2_Processor_ASVspoof_bce"
processor = Wav2Vec2Processor.from_pretrained(processor_save_path)
model = Wav2Vec2ForSequenceClassification.from_pretrained(checkpoints_path)
model = model.to(device)  # Ensure model is on GPU if available
model.eval()  # Set to evaluation mode

# Function to load audio, process it, and extract embeddings
def extract_embeddings(file_path):
    try:
        # Load the audio file with librosa at a sample rate of 16 kHz
        audio_input, _ = librosa.load(file_path, sr=16000)
    except Exception as e:
        print(f"Warning: Could not load {file_path}. Skipping. Error: {e}")
        return None, None

    # Prepare the input values for the fine-tuned wav2vec2 model
    input_values = processor(audio_input, sampling_rate=16000, return_tensors="pt", padding=True).input_values.to(device)

    # Extract embeddings using the model
    with torch.no_grad():
        frame_embeddings = model.wav2vec2(input_values).last_hidden_state  # Access wav2vec2 model part directly

    # Mean pooling across frames to get a single embedding vector per audio file
    pooled_embedding = frame_embeddings.mean(dim=1).squeeze()

    # Return the embedding and its shape
    return pooled_embedding.cpu().numpy(), pooled_embedding.shape

# Initialize list to store embeddings
embeddings_list = []

# Process files in the filtered test data order
for _, row in tqdm(test_data.iterrows(), total=len(test_data)):
    base_filename = row['path'].replace('/', '_').replace('.wav', '')
    file_path = os.path.join(test_folder, base_filename + ".wav")

    if not os.path.exists(file_path):
        continue

    # Extract the emotion label
    emotion = row['emotion']

    # Extract embeddings from the WAV file
    embeddings, embedding_shape = extract_embeddings(file_path)

    if embeddings is None:
        continue

    # Append embeddings, file path, label, and shape info to list
    embeddings_list.append({
        "file_path": base_filename,
        "folder": "test",
        "emotion_label": emotion,
        "embedding_shape": embedding_shape,
        "embedding": embeddings
    })

# Convert list to DataFrame
embeddings_df = pd.DataFrame(embeddings_list)

# Convert embeddings to lists for saving in a CSV
embeddings_df['embedding'] = embeddings_df['embedding'].apply(lambda x: x.tolist())

# Save the DataFrame to Google Drive
embeddings_csv_path = "/content/drive/MyDrive/IEMOCAP_embeddings_NEW_bce/iemocap_embeddings_test_NEW_bce.csv"
embeddings_df.to_csv(embeddings_csv_path, index=False)

# Display the DataFrame in Colab
print(embeddings_df)
print(f"Embeddings and metadata saved to Google Drive at {embeddings_csv_path}")


Using device: cuda


100%|██████████| 1507/1507 [01:57<00:00, 12.78it/s]


                                              file_path folder emotion_label  \
0     Session1_sentences_wav_Ses01F_script02_1_Ses01...   test           fru   
1     Session1_sentences_wav_Ses01F_script02_1_Ses01...   test           sur   
2     Session1_sentences_wav_Ses01F_script02_1_Ses01...   test           xxx   
3     Session1_sentences_wav_Ses01F_script02_1_Ses01...   test           fru   
4     Session1_sentences_wav_Ses01F_script02_1_Ses01...   test           neu   
...                                                 ...    ...           ...   
1502  Session5_sentences_wav_Ses05M_script03_2_Ses05...   test           ang   
1503  Session5_sentences_wav_Ses05M_script03_2_Ses05...   test           ang   
1504  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...   test           sad   
1505  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...   test           sad   
1506  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...   test           sad   

     embedding_shape                   